# 09 - Reservation Policy Mechanism Checks

This notebook checks what is driving the strict reservation results, focusing on class-specific losses and reserved/general slot use.

The goal is to verify how strict Class 1 reservation changes access and losses relative to the shared FCFS mechanics already explored in earlier notebooks.


## Setup

Use the same simple repo-root detection and imports as notebook 07.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists() and (
            candidate / "analysis" / "metrics.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")

## Base Scenario

The baseline uses `lambda_1 = 25`, `lambda_2 = 25`, and `Q = 10`.

In [ ]:
BASE_SCENARIO = {
    "slots_per_day": 32,
    "reserved_slots_per_day": 10,
    "reserved_class_id": 1,
    "horizon_days": 14,
    "burn_in_days": 30,
    "measure_days": 365,
    "cooldown_days": 14,
    "seeds": list(range(5101, 5131)),
    "classes": {
        1: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
        2: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
    },
}

POLICIES = ["Strict C1 reservation"]
POLICY_COLORS = {
    "Strict C1 reservation": "tab:blue",
}
LOST_COMPONENTS = [
    "balked_rate",
    "canceled_rate",
    "no_show_rate",
    "no_offer_rate",
    "unresolved_booked_rate",
]
OUTCOME_STACK = [
    ("served_rate", "served", "#4c78a8"),
    ("balked_rate", "balked", "#f58518"),
    ("canceled_rate", "canceled", "#54a24b"),
    ("no_show_rate", "no-show", "#e45756"),
    ("no_offer_rate", "no-offer", "#b279a2"),
    ("unresolved_booked_rate", "unresolved booked", "#bab0ac"),
]

scenario_table = pd.DataFrame(
    {
        "value": pd.Series(
            {
                key: value
                for key, value in BASE_SCENARIO.items()
                if key not in {"classes", "seeds"}
            },
            dtype="object",
        )
    }
)
scenario_table.loc["num_seeds", "value"] = len(BASE_SCENARIO["seeds"])
scenario_table.loc["seed_range", "value"] = f"{BASE_SCENARIO['seeds'][0]}-{BASE_SCENARIO['seeds'][-1]}"

class_table = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "lambda_per_day": params["lambda_per_day"],
            "cancel_prob": params["cancel_prob"],
            "balk_threshold": params["balk_prob"]["threshold"],
            "balk_low": params["balk_prob"]["low"],
            "balk_high": params["balk_prob"]["high"],
            "no_show_threshold": params["no_show_prob"]["threshold"],
            "no_show_low": params["no_show_prob"]["low"],
            "no_show_high": params["no_show_prob"]["high"],
        }
        for class_id, params in BASE_SCENARIO["classes"].items()
    ]
)

display(scenario_table)
display(class_table)

## Small Helpers

The helpers are intentionally narrow: build configs, update demand, run strict reservation, add outcome shares, and validate accounting.


In [ ]:
def build_config(scenario: dict, policy: str, seed: int | None) -> SimulationConfig:
    classes = {}
    for class_id, params in scenario["classes"].items():
        classes[class_id] = PatientClassParams(
            class_id=class_id,
            lambda_per_day=float(params["lambda_per_day"]),
            balk_prob=ThresholdRule(**params["balk_prob"]),
            cancel_prob=float(params["cancel_prob"]),
            no_show_prob=ThresholdRule(**params["no_show_prob"]),
            value=float(params.get("value", 1.0)),
        )

    reserved_slots = int(scenario["reserved_slots_per_day"])

    return SimulationConfig(
        slots_per_day=int(scenario["slots_per_day"]),
        horizon_days=int(scenario["horizon_days"]),
        burn_in_days=int(scenario["burn_in_days"]),
        measure_days=int(scenario["measure_days"]),
        cooldown_days=int(scenario["cooldown_days"]),
        classes=classes,
        seed=seed,
        reserved_class_id=scenario["reserved_class_id"] if reserved_slots > 0 else None,
        reserved_slots_per_day=reserved_slots,
    )


def make_scenario(lambda_1: float, lambda_2: float, q: int = 10, base_scenario: dict = BASE_SCENARIO) -> dict:
    scenario = deepcopy(base_scenario)
    scenario["reserved_slots_per_day"] = int(q)
    scenario["classes"][1]["lambda_per_day"] = float(lambda_1)
    scenario["classes"][2]["lambda_per_day"] = float(lambda_2)
    return scenario


def add_rates(aggregate_df: pd.DataFrame, class_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    aggregate_df = aggregate_df.copy()
    class_df = class_df.copy()

    arrivals = aggregate_df["total_arrivals"]
    aggregate_df["served_rate"] = aggregate_df["total_served"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["balked_rate"] = aggregate_df["total_balked"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["no_offer_rate"] = aggregate_df["total_no_offer"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["canceled_rate"] = aggregate_df["total_canceled"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["no_show_rate"] = aggregate_df["total_no_show"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["unresolved_booked_rate"] = aggregate_df["total_unresolved_booked"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["lost_components_sum"] = aggregate_df[LOST_COMPONENTS].sum(axis=1)
    aggregate_df["lost_rate"] = 1.0 - aggregate_df["served_rate"]

    class_df["unresolved_booked"] = class_df["booked"] - class_df["canceled"] - class_df["no_show"] - class_df["served"]
    class_arrivals = class_df["arrivals"]
    class_df["served_rate"] = class_df["percent_serviced"]
    class_df["balked_rate"] = class_df["balked"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["no_offer_rate"] = class_df["no_offer"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["canceled_rate"] = class_df["canceled"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["no_show_rate"] = class_df["no_show"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["unresolved_booked_rate"] = class_df["unresolved_booked"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["lost_components_sum"] = class_df[LOST_COMPONENTS].sum(axis=1)
    class_df["lost_rate"] = 1.0 - class_df["served_rate"]

    return aggregate_df, class_df


def validate_accounting(aggregate_df: pd.DataFrame, class_df: pd.DataFrame, tol: float = 1e-9) -> None:
    aggregate_partition = (
        aggregate_df["total_served"]
        + aggregate_df["total_balked"]
        + aggregate_df["total_no_offer"]
        + aggregate_df["total_canceled"]
        + aggregate_df["total_no_show"]
        + aggregate_df["total_unresolved_booked"]
    )
    if (aggregate_partition - aggregate_df["total_arrivals"]).abs().max() > tol:
        raise AssertionError("Aggregate outcomes do not partition arrivals.")
    if (aggregate_df["total_unresolved_booked"] < -tol).any():
        raise AssertionError("Aggregate unresolved_booked is negative.")
    if (aggregate_df["lost_rate"] - (1.0 - aggregate_df["served_rate"])).abs().max() > tol:
        raise AssertionError("Aggregate lost_rate is not 1 - served_rate.")
    if (aggregate_df["lost_components_sum"] - aggregate_df["lost_rate"]).abs().max() > tol:
        raise AssertionError("Aggregate lost components do not sum to lost_rate.")

    class_partition = (
        class_df["served"]
        + class_df["balked"]
        + class_df["no_offer"]
        + class_df["canceled"]
        + class_df["no_show"]
        + class_df["unresolved_booked"]
    )
    if (class_partition - class_df["arrivals"]).abs().max() > tol:
        raise AssertionError("Class outcomes do not partition arrivals.")
    if (class_df["unresolved_booked"] < -tol).any():
        raise AssertionError("Class unresolved_booked is negative.")
    if (class_df["lost_rate"] - (1.0 - class_df["served_rate"])).abs().max() > tol:
        raise AssertionError("Class lost_rate is not 1 - served_rate.")
    if (class_df["lost_components_sum"] - class_df["lost_rate"]).abs().max() > tol:
        raise AssertionError("Class lost components do not sum to lost_rate.")


def run_policies(scenario: dict, case: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    aggregate_rows = []
    class_rows = []

    for policy in tqdm(POLICIES, desc=case):
        for seed in scenario["seeds"]:
            config = build_config(scenario, policy, seed=int(seed))
            result = ClinicAppointmentSimulation(config).run()
            fixed_values = {
                "case": case,
                "policy": policy,
                "seed": int(seed),
                "lambda_1": scenario["classes"][1]["lambda_per_day"],
                "lambda_2": scenario["classes"][2]["lambda_per_day"],
                "Q": scenario["reserved_slots_per_day"],
            }
            aggregate_rows.append(aggregate_result_row(result, fixed_values))
            class_rows.extend(class_result_rows(result, fixed_values))

    aggregate_df = pd.DataFrame(aggregate_rows)
    class_df = pd.DataFrame(class_rows)
    aggregate_df, class_df = add_rates(aggregate_df, class_df)
    validate_accounting(aggregate_df, class_df)
    return aggregate_df, class_df

## Baseline Run

Run strict reservation at `lambda_1 = 25`, `lambda_2 = 25`, and `Q = 10`.


In [ ]:
baseline_scenario = make_scenario(lambda_1=25, lambda_2=25, q=10)
baseline_aggregate_df, baseline_class_df = run_policies(baseline_scenario, case="baseline")

baseline_aggregate_summary = (
    baseline_aggregate_df.groupby("policy")[["served_rate", "average_utilization", "mean_offered_booking_delay", "lost_rate"]]
    .mean()
    .round(4)
)
baseline_class_summary = (
    baseline_class_df.groupby(["policy", "class_id"])[["served_rate", "mean_offered_booking_delay", "lost_rate"]]
    .mean()
    .round(4)
)

display(baseline_aggregate_summary)
display(baseline_class_summary)

## Class-Specific Outcome Decomposition

Each stacked bar is a mean share per measured arrival, averaged across seeds.

In [ ]:
outcome_columns = [column for column, _, _ in OUTCOME_STACK]

baseline_decomp = (
    baseline_class_df.groupby(["policy", "class_id"])[outcome_columns]
    .mean()
    .reset_index()
)
baseline_decomp["bar_label"] = baseline_decomp.apply(
    lambda row: "Strict
Class " + str(int(row["class_id"])),
    axis=1,
)
baseline_decomp = baseline_decomp.sort_values(["policy", "class_id"])

display(baseline_decomp[["policy", "class_id", *outcome_columns]].round(4))

fig, ax = plt.subplots(figsize=(8, 5.5))
bottom = pd.Series(0.0, index=baseline_decomp.index)
for column, label, color in OUTCOME_STACK:
    ax.bar(
        baseline_decomp["bar_label"],
        baseline_decomp[column],
        bottom=bottom,
        label=label,
        color=color,
    )
    bottom = bottom + baseline_decomp[column]

ax.set_title("Baseline outcome decomposition by class")
ax.set_ylabel("Mean share per measured arrival")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()


## Demand Stress Outcome Decomposition

Repeat the decomposition with Class 2 demand at `lambda_2 = 80`, while Class 1 remains at `lambda_1 = 25`. This checks how strict reservation behaves when Class 2 demand is extreme.


In [ ]:
stress_scenario = make_scenario(lambda_1=25, lambda_2=80, q=10)
stress_aggregate_df, stress_class_df = run_policies(stress_scenario, case="class_2_overload")

stress_aggregate_summary = (
    stress_aggregate_df.groupby("policy")[["served_rate", "average_utilization", "mean_offered_booking_delay", "lost_rate"]]
    .mean()
    .round(4)
)
stress_class_summary = (
    stress_class_df.groupby(["policy", "class_id"])[["served_rate", "mean_offered_booking_delay", "lost_rate"]]
    .mean()
    .round(4)
)

display(stress_aggregate_summary)
display(stress_class_summary)

In [ ]:
stress_decomp = (
    stress_class_df.groupby(["policy", "class_id"])[outcome_columns]
    .mean()
    .reset_index()
)
stress_decomp["bar_label"] = stress_decomp.apply(
    lambda row: "Strict
Class " + str(int(row["class_id"])),
    axis=1,
)
stress_decomp = stress_decomp.sort_values(["policy", "class_id"])

display(stress_decomp[["policy", "class_id", *outcome_columns]].round(4))

fig, ax = plt.subplots(figsize=(8, 5.5))
bottom = pd.Series(0.0, index=stress_decomp.index)
for column, label, color in OUTCOME_STACK:
    ax.bar(
        stress_decomp["bar_label"],
        stress_decomp[column],
        bottom=bottom,
        label=label,
        color=color,
    )
    bottom = bottom + stress_decomp[column]

ax.set_title("Class 2 overload outcome decomposition by class")
ax.set_ylabel("Mean share per measured arrival")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()


## Reserved/General Slot Usage Diagnostic

Reserved/general slot usage is not yet exposed cleanly by the metrics layer. `Booking` objects inside the engine have a `reserved_slot` flag, but `SimulationResults` does not retain a measured booking audit or expose `result.calendar`. The public `final_full_state` view stores only `(patient_class, booking_delay)` tuples, so it loses whether each booking used reserved or general capacity.

This should be added if this mechanism becomes central to the report.

In [ ]:
sample_config = build_config(baseline_scenario, "Strict C1 reservation", seed=BASE_SCENARIO["seeds"][0])
sample_result = ClinicAppointmentSimulation(sample_config).run()
first_nonempty_slot = next(
    (slot for day in sample_result.final_full_state for slot in day if slot != 0),
    None,
)

print("SimulationResults exposes calendar:", hasattr(sample_result, "calendar"))
print("Example final_full_state booking:", first_nonempty_slot)
print("final_full_state booking has reserved_slot flag:", hasattr(first_nonempty_slot, "reserved_slot"))


## Accounting Checks

Validate outcome partitioning for the baseline and stress runs.

In [ ]:
for aggregate_df, class_df, label in [
    (baseline_aggregate_df, baseline_class_df, "baseline"),
    (stress_aggregate_df, stress_class_df, "class_2_overload"),
]:
    validate_accounting(aggregate_df, class_df)
    print(f"Accounting checks passed: {label}")

## Short Interpretation Placeholders

- Baseline:
- Class 2 overload stress:
- Reserved/general slot usage:
- Mechanism conclusion: